In [ ]:
# Install required libraries for Phase 2
!pip install -q langchain langchain-community langchain-huggingface sentence-transformers chromadb

from google.colab import drive
import os

# Mount Drive to access the Knowledge Base we built in Phase 1
drive.mount('/content/drive')

# Verify the database exists
db_path = '/content/drive/My Drive/ZTA_Project/chroma_db_bge'
if os.path.exists(db_path):
    print("Developer Log: Database successfully located on Google Drive.")
else:
    print("WARNING: Database not found. Check the file path.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 133.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 130.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6

In [ ]:
%%writefile retrieval.py
import torch
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import Chroma
from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

class AdvancedRetriever:
    def __init__(self, db_path):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Initializing Advanced Retriever on {self.device.upper()}...")

        # 1. Load the Bi-Encoder (BGE Embeddings from Phase 1)
        self.embeddings = HuggingFaceBgeEmbeddings(
            model_name="BAAI/bge-large-en-v1.5",
            model_kwargs={'device': self.device},
            encode_kwargs={'normalize_embeddings': True}
        )

        # Load ChromaDB
        self.db = Chroma(persist_directory=db_path, embedding_function=self.embeddings)

        # 2. Load the Cross-Encoder for Reranking (Post-Retrieval Filter)
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=self.device)

        # 3. Load Query Rewriter (Pre-Retrieval)
        # FIX: Bypassing the 'pipeline' entirely and loading the model explicitly
        print("Loading Query Rewriter directly...")
        self.tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
        self.rewriter_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").to(self.device)

    def rewrite_query(self, original_query):
        """TECHNIQUE 1: Query Rewriting (HyDE concept)"""
        prompt = f"Rewrite this user question into a technical cybersecurity search query focusing on Zero Trust Architecture: {original_query}"

        # Generate the expanded query using raw tokenization
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        outputs = self.rewriter_model.generate(**inputs, max_length=50)
        expanded_query = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        return expanded_query

    def retrieve_and_rerank(self, user_query, top_k_initial=5, top_k_final=3):
        """TECHNIQUE 2 & 3: Bi-Encoder Retrieval and Cross-Encoder Reranking"""

        # Step 1: Rewrite the query
        search_query = self.rewrite_query(user_query)

        # Step 2: Vector Search (The Broad Net)
        initial_docs = self.db.similarity_search(search_query, k=top_k_initial)

        # Step 3: Reranking (The Microscope)
        pairs = [[user_query, doc.page_content] for doc in initial_docs]
        scores = self.reranker.predict(pairs)

        scored_docs = list(zip(initial_docs, scores))
        scored_docs.sort(key=lambda x: x[1], reverse=True)

        final_docs = [doc for doc, score in scored_docs[:top_k_final]]

        return {
            "original_query": user_query,
            "rewritten_query": search_query,
            "documents": final_docs
        }

Writing retrieval.py


In [ ]:
# Import the module we just created
from retrieval import AdvancedRetriever

# Initialize the engine using the path to your Google Drive DB
db_path = '/content/drive/My Drive/ZTA_Project/chroma_db_bge'
retriever = AdvancedRetriever(db_path)

# Simulate a High-Level Architect Query
test_query = "How do I prevent lateral movement if a remote worker's laptop is compromised?"

print("\n--- Executing Advanced Retrieval Pipeline ---\n")
results = retriever.retrieve_and_rerank(test_query)

print(f"Original Query:  {results['original_query']}")
print(f"Rewritten Query: {results['rewritten_query']}\n")

print("--- Top 3 Reranked Context Chunks ---")
for i, doc in enumerate(results['documents']):
    source = doc.metadata.get('source_document', 'Unknown Source')
    print(f"\n[Result {i+1} | Source: {source}]")
    print(doc.page_content[:400] + "...\n")

Initializing Advanced Retriever on CUDA...


/content/retrieval.py:13: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceBgeEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


--- Executing Advanced Retrieval Pipeline ---

Original Query:  How do I prevent lateral movement if a remote worker's laptop is compromised?
Rewritten Query: What is the best way to prevent lateral movement if a remote worker's laptop is compromised?

--- Top 3 Reranked Context Chunks ---

[Result 1 | Source: NIST_800-207]
collection of resources once on the internal network. As a result, unauthorized lateral movement 
within the environment has been one of the biggest challenges for federal agencies.  
The Trusted Internet Connections (TIC) and agency perimeter firewalls provide strong internet 
gateways. This helps block attackers from the internet, but the TICs and perimeter firewalls are 
less useful for detecti...


[Result 2 | Source: NIST_800-53]
Use lockable physical casings to protect [Assignment: organization-defined system 
components] from unauthorized physical access. 
Discussion:  The greatest risk from the use of portable devices— such as smart phones, 
tablets, and no

In [ ]:
import shutil
import os

# Define the source and destination
source_file = 'retrieval.py'
drive_destination = '/content/drive/My Drive/ZTA_Project/retrieval.py'

# Copy the file to your Google Drive project folder
shutil.copy(source_file, drive_destination)

print(f"Developer Log: Code successfully backed up to {drive_destination}")

Developer Log: Code successfully backed up to /content/drive/My Drive/ZTA_Project/retrieval.py


### Phase 2 Summary: The Advanced Search Engine

In this phase, I upgraded the system from a basic vector database into an **Advanced RAG** pipeline. I built a modular retrieval engine (`retrieval.py`) designed to filter out noise and pull only the most accurate "ground truth" documents for the LLM.

**Key Techniques Implemented:**

* **Query Rewriting (Pre-Retrieval):** I integrated a lightweight sequence-to-sequence model (`google/flan-t5-small`) to automatically translate the user's raw question into a targeted search query. This helps the system search based on technical intent rather than just matching random words.
* **Bi-Encoder Search (Vector Retrieval):** I used the BGE embedding model to query the ChromaDB I built in Phase 1. This acts as a broad net, quickly scanning all 4,200+ chunks to grab the Top-5 most semantically similar paragraphs.
* **Cross-Encoder Reranking (Post-Retrieval Filter):** I implemented a dedicated reranking model (`ms-marco-MiniLM-L-6-v2`). This acts as a strict filter. It reads the 5 retrieved chunks alongside the user's original query, scores them based on true logical relevance, and discards the bottom 2.

**Result:** The pipeline is fully operational. It successfully takes a user query, translates it, searches the database, and outputs the absolute best Top-3 context chunks, complete with source metadata. This clean data package is now ready to be sent to the generation model in Phase 3.